# Predictive Anayltics: Support Vector Machines

Approach for SVM:
– Simply start without a kernel. Then, gradually make your model complex by integrating different
kind of kernels. Also, use grid search to find optimal values for your hyperparameters.
– How good is your model? Evaluate your model’s performance and comment on its shortfalls.
– Show how you model’s performance varies as you increase or decrease temporal or spatial
resolution How does your performance change when you only use census tract as spatial units?
– How could the model be improved further? Explain some of the improvement levers that you might
focus on in a follow-up project.

In [10]:
DO_GRID_SEARCH = True
GRID_SAMPLE = 500
SPATIAL_UNIT = "census" # options: census, hexa, community

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib as plt
import datetime
from sklearn.svm import SVC 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

## Preparations

In [12]:
# "Settings" / Decisions for the training data

# "Settings" / Decisions for the training data
if SPATIAL_UNIT == "census":    
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "hexa":
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
elif SPATIAL_UNIT == "community": 
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 
else:
    print("Warning: No type of Spatial Unit given, Used census tract")
    DATA_PATH_TRAIN = "../data/train_test_data/svm_census_train.parquet"
    DATA_PATH_VAL = "../data/train_test_data/svm_census_val.parquet"
    DATA_PATH_TEST = "../data/train_test_data/svm_census_test.parquet" 


MODEL_PATH = "../models"

# Target and feature selection
TARGET_COL = "trip_demand"
EXCLUDE_COLS = [
    TARGET_COL,
    "datetime_hour",      # real timestamp would lead to much leakage
    "_split_bucket",      # only used for splitting
    "month", # cyclic feature is used instead
    "weekday", # cyclic feature is used instead
    "hour", # cyclic feature is used instead
    # all other taxi data columns must be excluded to prevent leakage
    "trip_seconds_sum",
    "trip_seconds_mean",
    "trip_seconds_min",
    "trip_seconds_max",
    "trip_miles_sum",
    "trip_miles_mean",
    "trip_miles_min",
    "trip_miles_max",
    "fare_sum",
    "fare_mean",
    "fare_min",
    "fare_max",
    "tips_sum",
    "tips_mean",
    "tips_min",
    "tips_max",
    "tolls_sum",
    "tolls_mean",
    "tolls_min",
    "tolls_max",
    "extras_sum",
    "extras_mean",
    "extras_min",
    "extras_max",
    "trip_total_sum",
    "trip_total_mean",
    "trip_total_min",
    "trip_total_max",
    "most_common_payment_type",
    "trip_count",
    "date",
]

Load data and select features and target

In [13]:
# Load data
train = pl.scan_parquet(DATA_PATH_TRAIN)
val = pl.scan_parquet(DATA_PATH_VAL)
test = pl.scan_parquet(DATA_PATH_TEST)

In [14]:
train_df = train.collect()
val_df = val.collect()
test_df = test.collect()

train_df = train_df.to_pandas()
val_df = val_df.to_pandas()
test_df = test_df.to_pandas()

In [6]:
train_df.head()
type(train_df)

pandas.core.frame.DataFrame

In [ ]:
# prepare data
# calculate median to split in low/high demand
# when trip_count above 50 percent use "high", when below or equal to 50 percent low


# definition for demand cause currently the q1 is 0, q2 is 1 and q3 is 3
# I excluded the zeros since a magority of values are zero
p90 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 90)
p70 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 70)
p50 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 50)
p25 = np.percentile(test_df.loc[test_df["trip_count"] > 0, "trip_count"], 25)

train_df["trip_demand"] = "Low"
train_df.loc[train_df["trip_count"] >= p25, "trip_demand"] = "Mid"
train_df.loc[train_df["trip_count"] >= p50, "trip_demand"] = "Mid High"
train_df.loc[train_df["trip_count"] >= p70, "trip_demand"] = "High"
train_df.loc[train_df["trip_count"] >= p90, "trip_demand"] = "Very High"


p90 = np.percentile(val_df["trip_count"], 90)
p70 = np.percentile(val_df["trip_count"], 70)

val_df["trip_demand"] = "Low"
train_df.loc[train_df["trip_count"] >= p25, "trip_demand"] = "Mid"
train_df.loc[train_df["trip_count"] >= p50, "trip_demand"] = "Mid High"
train_df.loc[train_df["trip_count"] >= p70, "trip_demand"] = "High"
train_df.loc[train_df["trip_count"] >= p90, "trip_demand"] = "Very High"

p90 = np.percentile(test_df["trip_count"], 90)
p70 = np.percentile(test_df["trip_count"], 70)

test_df["trip_demand"] = "Low"
train_df.loc[train_df["trip_count"] >= p25, "trip_demand"] = "Mid"
train_df.loc[train_df["trip_count"] >= p50, "trip_demand"] = "Mid High"
train_df.loc[train_df["trip_count"] >= p70, "trip_demand"] = "High"
train_df.loc[train_df["trip_count"] >= p90, "trip_demand"] = "Very High"


In [33]:
print(test_df.loc[test_df["trip_count"] == 0, "trip_count"].count())
print(test_df.loc[ test_df["trip_count"] > 0, "trip_count"].count())

2663489
31205


In [ ]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

# Community_area is a categorical id, not a numeric quantity, so one-hot encode it
X_train = pd.get_dummies(train_df[feature_cols], columns=["community_area"])
X_val = pd.get_dummies(val_df[feature_cols], columns=["community_area"])
X_test = pd.get_dummies(test_df[feature_cols], columns=["community_area"])

# Keep the dummy columns before scaling turns X_train into a plain array
train_columns = X_train.columns

# Make sure val/test have the same dummy columns as train (in case a community_area is missing)
X_val = X_val.reindex(columns=train_columns, fill_value=0)
X_test = X_test.reindex(columns=train_columns, fill_value=0)

y_train = train_df[TARGET_COL]
y_val = val_df[TARGET_COL]
y_test = test_df[TARGET_COL]

# SVR is distance-based, so all features need to be on a similar scale
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print("Features:", feature_cols)
print("Target:", y_train.dtypes)

In [ ]:
train_df = train_df.sample(n=5000, random_state=42)

In [ ]:
# Create X and y
feature_cols = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

X_train = train_df[feature_cols]
y_train = train_df[TARGET_COL]

X_val = val_df[feature_cols]
y_val = val_df[TARGET_COL]

X_test = test_df[feature_cols]
y_test = test_df[TARGET_COL]


print("Features:", X_train.dtypes)
print("Target:", y_train.dtypes)

Features: month_sin         float64
month_cos         float64
weekday_sin       float64
weekday_cos       float64
hour_sin          float64
hour_cos          float64
tmpc              float64
relh              float64
sknt              float64
vsby              float64
p01m              float64
skyc1_BKN            int8
skyc1_CLR            int8
skyc1_FEW            int8
skyc1_OVC            int8
skyc1_SCT            int8
skyc1_VV             int8
is_holiday           int8
community_area      int64
food_drink        float64
landmark          float64
shop              float64
train_station     float64
dtype: object
Target: uint32


In [ ]:
train_df

,datetime_hour,month,weekday,hour,month_sin,month_cos,weekday_sin,weekday_cos,hour_sin,hour_cos,...,extras_sum,extras_mean,extras_min,extras_max,trip_total_sum,trip_total_mean,trip_total_min,trip_total_max,most_common_payment_type,trip_demand
12153,2024-06-10 18:00:00,6,1,18,0.500000,-8.660254e-01,0.000000,1.000000,-1.000000,-1.836970e-16,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
653944,2024-05-15 11:00:00,5,3,11,0.866025,-5.000000e-01,0.974928,-0.222521,0.258819,-9.659258e-01,...,0.0,0.000000,0.0,0.0,66.25,33.125000,29.25,37.00,Prcard,high
1073833,2025-02-17 01:00:00,2,1,1,0.500000,8.660254e-01,0.000000,1.000000,0.258819,9.659258e-01,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
705090,2024-10-19 10:00:00,10,6,10,-1.000000,-1.836970e-16,-0.974928,-0.222521,0.500000,-8.660254e-01,...,0.0,0.000000,0.0,0.0,40.00,20.000000,9.75,30.25,Prcard,high
982471,2025-02-18 00:00:00,2,2,0,0.500000,8.660254e-01,0.781831,0.623490,0.000000,1.000000e+00,...,17.0,17.000000,17.0,17.0,43.00,43.000000,43.00,43.00,Cash,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
818913,2025-06-19 00:00:00,6,4,0,0.500000,-8.660254e-01,0.433884,-0.900969,0.000000,1.000000e+00,...,0.0,0.000000,0.0,0.0,0.00,0.000000,0.00,0.00,No trips,low
1061565,2024-02-28 08:00:00,2,3,8,0.500000,8.660254e-01,0.974928,-0.222521,0.866025,-5.000000e-01,...,31.0,0.146919,0.0,8.0,2606.17,12.351517,3.50,57.81,Credit Card,high
795342,2025-02-01 14:00:00,2,6,14,0.500000,8.660254e-01,-0.974928,-0.222521,-0.500000,-8.660254e-01,...,0.0,0.000000,0.0,0.0,64.50,32.250000,32.25,32.25,Unknown,high
887040,2024-02-04 07:00:00,2,7,7,0.500000,8.660254e-01,-0.781831,0.623490,0.965926,-2.588190e-01,...,0.0,0.000000,0.0,0.0,114.22,57.110000,40.00,74.22,Prcard,high


In [ ]:
model_svr = SVR()

In [ ]:
param_grid_linear = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["linear"]
}

param_grid_rbf_sigmoid = {
    "C": [0.1, 1, 10],
    "epsilon": [0.1, 0.5, 1, 1.5],
    "kernel": ["rbf", "sigmoid"],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

param_grid_poly = {
    "C": [0.1, 1, 10],
    "epsilon": [0.01, 0.1, 0.5, 1],
    "kernel": ["poly"],
    "degree": [3, 4],
    "gamma": ["scale", "auto", 0.01, 0.1, 1]
}

grids = {}
for name, grid in [("linear", param_grid_linear), ("rbf_sigmoid", param_grid_rbf_sigmoid), ("poly", param_grid_poly)]:
    search = GridSearchCV(
        estimator=SVR(),
        param_grid=grid,
        cv=3,
        scoring="r2",
        n_jobs=-1,
        error_score="raise"
    )
    search.fit(X_train_grid, y_train_grid)
    grids[name] = search
    print(name, "best score:", search.best_score_, "best params:", search.best_params_)

best_name = max(grids, key=lambda name: grids[name].best_score_)
grid_search = grids[best_name]
print("Overall best:", best_name, grid_search.best_params_)

c:\Users\bkran\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
# Best parameters
print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

Best parameters: {'C': 100, 'gamma': 'scale', 'kernel': 'rbf'}
Best CV score: 0.7689999999999999


In [ ]:
# Evaluate on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

#rint("Test accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Train SVC 

grid_search.fit(X_train, y_train, best_model=best_model)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [ ]:
# Make prediction 
y_pred = grid_search.predict(X_test)

In [ ]:
y_pred

array(['low', 'low', 'low', ..., 'low', 'low', 'low'],
      shape=(223531,), dtype=object)

In [ ]:
# Evaluate Model
print(confusion_matrix(y_test,y_pred))

print(classification_report(y_test,y_pred))

[[ 35931  36114]
 [ 13210 138276]]
              precision    recall  f1-score   support

        high       0.73      0.50      0.59     72045
         low       0.79      0.91      0.85    151486

    accuracy                           0.78    223531
   macro avg       0.76      0.71      0.72    223531
weighted avg       0.77      0.78      0.77    223531



In [ ]:
# get support vectors
grid_search.support_vectors_

array([[ 8.66025404e-01, -5.00000000e-01,  9.74927912e-01, ...,
         0.00000000e+00,  7.00000000e+00,  1.00000000e+00],
       [-1.00000000e+00, -1.83697020e-16, -9.74927912e-01, ...,
         1.20000000e+01,  1.30000000e+01,  4.00000000e+00],
       [ 8.66025404e-01, -5.00000000e-01,  9.74927912e-01, ...,
         3.50000000e+01,  9.50000000e+01,  1.60000000e+01],
       ...,
       [ 1.22464680e-16, -1.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  5.00000000e+00,  0.00000000e+00],
       [-1.00000000e+00, -1.83697020e-16, -4.33883739e-01, ...,
         8.00000000e+00,  2.80000000e+01,  6.00000000e+00],
       [ 5.00000000e-01, -8.66025404e-01,  7.81831482e-01, ...,
         0.00000000e+00,  2.00000000e+01,  2.00000000e+00]],
      shape=(599, 23))